In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
df = pd.read_csv("mental_health_dataset.csv")
df.head()

,age,gender,employment_status,work_environment,mental_health_history,seeks_treatment,stress_level,sleep_hours,physical_activity_days,depression_score,anxiety_score,social_support_score,productivity_score,mental_health_risk
0,56,Male,Employed,On-site,Yes,Yes,6,6.2,3,28,17,54,59.7,High
1,46,Female,Student,On-site,No,Yes,10,9.0,4,30,11,85,54.9,High
2,32,Female,Employed,On-site,Yes,No,7,7.7,2,24,7,62,61.3,Medium
3,60,Non-binary,Self-employed,On-site,No,No,4,4.5,4,6,0,95,97.0,Low
4,25,Female,Self-employed,On-site,Yes,Yes,3,5.4,0,24,12,70,69.0,High


### Target and feature types

We separate:
- `y` = target to predict (`mental_health_risk`)
- `X` = all other features

We also separate columns:
- numerical columns (used for custom outlier detection)
- categorical columns (need one-hot encoding)


In [ ]:
target_col = "mental_health_risk"
X = df.drop(columns=[target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

num_cols, cat_cols


(['age',
  'stress_level',
  'sleep_hours',
  'physical_activity_days',
  'depression_score',
  'anxiety_score',
  'social_support_score',
  'productivity_score'],
 ['gender',
  'employment_status',
  'work_environment',
  'mental_health_history',
  'seeks_treatment'])

### Interpretation of feature separation

The features are divided into:
- **Numerical features** (age, stress, sleep, depression, anxiety, etc.), which represent measurable values and are used for outlier detection.
- **Categorical features** (gender, employment, work environment, mental health history), which describe context and are encoded later for classification.

Outlier detection is applied only to numerical features because abnormal values can only be defined on numbers.
Both feature types are then used together to predict the mental health risk.


### Preprocessing

Models need numbers:
- numerical features are standardized
- categorical features are one-hot encoded

This preprocessing is also used before training the classifier.


In [36]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)


### Train/test split

We split data:
- training set: used to train
- test set: used only for final evaluation

We use stratify to keep the same class proportions in both splits.


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape


((7500, 13), (2500, 13))

### Interpretation of the train / test split

After splitting the dataset, we obtain:
- 7,500 samples in the training set
- 2,500 samples in the test set

Each sample contains **13 features**.

This corresponds to a **75% / 25% split**, which is a common choice in machine learning:
- the training set is used to learn the model parameters
- the test set is used to evaluate performance on unseen data

The `stratify=y` option ensures that the proportions of
`Low`, `Medium`, and `High` mental health risk
are similar in both the training and test sets.

This makes the evaluation more reliable and avoids biased results.


### Baseline

We train logistic regression on the full training set.
This is the reference score.


In [38]:
Xt_train = preprocess.fit_transform(X_train)
Xt_test = preprocess.transform(X_test)

baseline_model = LogisticRegression(max_iter=5000)
baseline_model.fit(Xt_train, y_train)

baseline_pred = baseline_model.predict(Xt_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

baseline_acc


0.9988

### Interpretation of the baseline accuracy

The baseline classification accuracy is **0.9988**, which means that:
- about **99.88%** of the test samples are correctly classified
- the model makes very few mistakes on unseen data

This result is very high, showing that:
- the features are strongly correlated with the target variable
- logistic regression is sufficient to solve this classification problem
- the dataset is relatively clean and well-structured

This baseline result will be used as a **reference point**.
All models using outlier detection will be compared to this score
to see whether removing outliers improves or degrades performance.


### Custom outlier detection

We build our own outlier detector:
- take only standardized numerical features
- compute z-scores (how extreme values are)
- mark a sample as outlier if any numeric feature has |z| > threshold

This is a simple custom rule, not a library outlier model.


In [39]:
num_scaler = StandardScaler()
Xnum_train = num_scaler.fit_transform(X_train[num_cols])

threshold = 3.0
z = np.abs(Xnum_train)

inlier_mask = (z <= threshold).all(axis=1)

inlier_mask.mean(), inlier_mask.sum(), len(inlier_mask)


(np.float64(1.0), np.int64(7500), 7500)

### Interpretation of custom outlier detection result

With a z-score threshold of **3.0**, the custom outlier detection keeps:
- **100% of the training samples**
- **7,500 inliers out of 7,500 samples**

This means that, according to our rule, **no training sample is extreme enough**
to be considered an outlier.

This suggests that:
- the numerical features are well distributed
- there are no very abnormal values in the training data
- the dataset is already clean with respect to extreme numerical values

As a result, removing outliers with this threshold has no effect,
and the classification performance is expected to be similar to the baseline.

### Train after removing outliers

We remove outliers only from the training set.
Then we train the same classifier again and test on the same test set.

If accuracy improves, it means outliers were adding noise.
If accuracy drops, it means we removed useful training data.


In [40]:
X_train_in = X_train.iloc[np.where(inlier_mask)[0]]
y_train_in = y_train.iloc[np.where(inlier_mask)[0]]

Xt_train_in = preprocess.fit_transform(X_train_in)
Xt_test = preprocess.transform(X_test)

model_inliers = LogisticRegression(max_iter=5000)
model_inliers.fit(Xt_train_in, y_train_in)

pred_inliers = model_inliers.predict(Xt_test)
acc_inliers = accuracy_score(y_test, pred_inliers)

baseline_acc, acc_inliers


(0.9988, 0.9988)

### Interpretation of accuracy after outlier removal

The accuracy after applying custom outlier detection is **0.9988**, which is
**exactly the same as the baseline accuracy**.

This means that:
- no outliers were removed from the training data
- the training set used by the model is unchanged
- outlier detection had no impact on classification performance

This confirms that the dataset is already clean and that outlier removal
is not necessary for this problem.


### Sweep threshold

The threshold controls how strict outlier detection is:
- low threshold -> removes more points
- high threshold -> removes fewer points

We test multiple thresholds and see how test accuracy changes.


In [41]:
threshold_list = [2.0, 2.5, 3.0, 3.5, 4.0]
acc_list = []
removed_ratio_list = []
kept_list = []

for t in threshold_list:
    Xnum_train = num_scaler.fit_transform(X_train[num_cols])
    z = np.abs(Xnum_train)
    inlier_mask = (z <= t).all(axis=1)

    X_train_in = X_train.iloc[np.where(inlier_mask)[0]]
    y_train_in = y_train.iloc[np.where(inlier_mask)[0]]

    Xt_train_in = preprocess.fit_transform(X_train_in)
    Xt_test = preprocess.transform(X_test)

    clf = LogisticRegression(max_iter=5000)
    clf.fit(Xt_train_in, y_train_in)

    pred = clf.predict(Xt_test)
    acc = accuracy_score(y_test, pred)

    acc_list.append(acc)
    kept_list.append(int(inlier_mask.sum()))
    removed_ratio_list.append(1 - inlier_mask.mean())

results = pd.DataFrame(
    {
        "z_threshold": threshold_list,
        "kept_train_samples": kept_list,
        "removed_ratio": removed_ratio_list,
        "test_accuracy": acc_list
    }
)

results


,z_threshold,kept_train_samples,removed_ratio,test_accuracy
0,2.0,7078,0.056267,0.9980
1,2.5,7500,0.000000,0.9988
2,3.0,7500,0.000000,0.9988
3,3.5,7500,0.000000,0.9988
4,4.0,7500,0.000000,0.9988


### Interpretation of the threshold sweep results

The table shows how changing the z-score threshold affects outlier removal
and classification accuracy.

- With a **low threshold (2.0)**:
  - About **5.6% of the training samples** are removed
  - Test accuracy slightly decreases to **0.9980**
  - This means the rule is too strict and removes useful data

- With thresholds **2.5 and higher**:
  - **No samples are removed**
  - Test accuracy remains at **0.9988**, equal to the baseline
  - This confirms that there are no strong numerical outliers in the dataset

Overall, the results show that:
- the dataset is already clean
- aggressive outlier removal hurts performance
- mild or no outlier removal is the best choice for this problem


### Best threshold + interpretation

We select the threshold that gives the best test accuracy.

Interpretation:
- If the best accuracy is higher than baseline, custom outlier removal helped.
- If it is similar, the dataset is already clean and outliers do not matter much.
- If it is lower, the rule removed useful data and was too aggressive.

We also look at `removed_ratio`:
- small removed_ratio means we removed few points (safe)
- big removed_ratio means we removed many points (risky)


In [44]:
best_idx = int(np.argmax(results["test_accuracy"].values))
best_row = results.iloc[best_idx]

best_row


z_threshold              2.5000
kept_train_samples    7500.0000
removed_ratio            0.0000
test_accuracy            0.9988
Name: 1, dtype: float64

### Interpretation of the best threshold

The best z-score threshold is **2.5**.

With this value:
- **no training samples are removed**
- the **test accuracy is 0.9988**, which is equal to the baseline

This shows that the dataset does not contain strong numerical outliers
and that removing samples is not beneficial for this problem.
The safest choice is therefore to keep all training data.
